## 목표: '다중선형회귀 모델' 구현
### Dataset: ../data/robot_data.csv
알아두기: 단순 선형회귀와 달리 다중선형회귀는 특징을 여러 개 받아서 하나의 결과값을 예측함
*전체 흐름*
1. pandas로 CSV 불러오기
2. 데이터 확인
3. 입력 X, 정답 y 나누기
4. train/test 분리
5. Tensor로 변환
6. 모델 생성
7. loss, optimizer 설정
8. 학습
9. 예측
10. 평가

In [1]:
# PyTorch 관련 라이브러리
import torch
import torch.nn as nn
import torch.optim as optim

# 데이터 처리 라이브러리
import numpy as np
import pandas as pd

# 그래프 시각화 라이브러리
import matplotlib.pyplot as plt

# 학습 데이터 / 테스트 데이터 분리
from sklearn.model_selection import train_test_split

# 모델 평가 지표
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'현재 사용 중인 디바이스: {device}')

현재 사용 중인 디바이스: mps


In [4]:
# 1. pandas로 CSV 불러오기
df = pd.read_csv('../data/robot_data.csv')

In [14]:
# 2. 데이터 확인
df.head()

,speed,distance,battery,energy
0,4.37,6.41,71.36,19.17
1,9.56,33.64,26.73,37.31
2,7.59,19.15,32.93,32.34
3,6.39,27.89,91.88,37.18
4,2.40,45.84,68.51,30.52


In [ ]:
# 3. 입력 X, 정답 y 나누기

# X(속도+거리+배터리): 3가지 특성을 사용해 y('energy')를 예측
X = df[['speed', 'distance', 'battery']]
y = df[['energy']]

# 결과: (데이터수(행): 100개, 속성)
print(X.shape)
print(y.shape)

(100, 3)
(100, 1)


In [18]:
# 4. train/test 분리

# 평가 비율(8:2)
# 난수 시드는 자유입니다
# TMI. 42로 한 이유는 SF 소설 <은하수를 여행하는 히치하이커를 위한 안내서>에서 슈퍼 컴퓨터가 750만 년 동안 우주의 진리를 계산한 답이 42라고 하네요
# 그리고 증명은 안 됐지만 강사님이 다른 랜덤시드값보다 42로 했을 때 잘 나오는 것 같다는 아주 개인적인 소견도...내놓으셨습니다
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(80, 3)
(20, 3)
(80, 1)
(20, 1)


In [19]:
# 5. Tensor로 변환
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

print(type(X_train))
print(type(X_test))
print(type(y_train))
print(type(y_test))

<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [ ]:
# 6. 모델 생성